# DINOv3 ConvNeXt pyramid decoder — crop emergence counter (training)

Frozen DINOv3 ConvNeXt backbone -> pyramid decoder -> stride-4 peak heatmap
-> local-max decode -> **points + count**. All reusable code is the
`cropcounter` package (`pip install -e .` from the repo root):

- `cropcounter.crop_dataset` — CVAT / COCO-keypoints / Datumaro parsers, the
  train/val split loader, the native-resolution tile dataset
- `cropcounter.dinov3_pyramid` — frozen backbone + pyramid decoder
- `cropcounter.heatmap` — Gaussian target rendering + local-max peak decoding
- `cropcounter.losses`, `cropcounter.metrics`, `cropcounter.train`

Runs against the bundled anonymised sample set in `../examples/data` (20
images) so it works out of the box with no private data. Point it at your
own dataset by changing `cfg.data_root` below — see `examples/README.md`
for the expected `{train,val}/{annotations.xml, images/}` layout.

Run top to bottom from the `notebooks/` directory; training artifacts land
in `runs/<run_name>/` (`best.pt`, `last.pt`, `history.json`, `curves.png`),
gitignored.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from cropcounter import (
    CropTileDataset, TrainConfig, collate_val, decode_peaks, load_checkpoint,
    load_splits, match_points, render_targets, resolve_device, sweep_tau, train,
)
from cropcounter.dinov3_pyramid import IMAGENET_MEAN, IMAGENET_STD

device = resolve_device()
print(torch.__version__, device)

## Config

Load the recorded hyperparameters for the 13-epoch reference run, then repoint the data/weights/output paths at this repo's bundled sample set — the notebook is self-contained and doesn't need the private full dataset to run.

In [ ]:
cfg = TrainConfig.from_json(Path("../examples/config_13ep.json"))
cfg.data_root = Path("../examples/data")
cfg.weights_dir = Path("../weights")
cfg.out_dir = Path("runs")

assert cfg.data_root.is_dir(), f"missing {cfg.data_root.resolve()} — run from notebooks/"
cfg

## Data

In [ ]:
# Pre-split on disk: data_root/{train,val}/{annotations.xml, images/}.
train_recs, val_recs = load_splits(cfg.data_root, fmt=cfg.annotation_format)

print(f"total: {len(train_recs) + len(val_recs)} images, "
      f"{sum(len(r.points) for r in train_recs + val_recs)} points")
print(f"train: {len(train_recs)} images ({sum(len(r.points) for r in train_recs)} pts)")
print(f"val:   {len(val_recs)} images ({sum(len(r.points) for r in val_recs)} pts)")

## Sanity checks

Cheap checks that run before spending any compute on training.

In [ ]:
# One augmented training tile, its Gaussian target, and the points decoded
# straight back off the target. Train-mode CropTileDataset returns a plain
# (image, target, n_points) tuple, not the dict val mode uses below.
sanity_ds = CropTileDataset(
    train_recs, cfg.train_images_dir, train=True, tile=cfg.tile,
    output_stride=cfg.output_stride, sigma=cfg.sigma,
    tiles_per_image=cfg.tiles_per_image, scale_jitter=cfg.scale_jitter,
    exclude_label_statuses=cfg.exclude_label_statuses,
)
img_t, target, n_pts = sanity_ds[np.random.randint(len(sanity_ds))]
img = img_t.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
pts, _ = decode_peaks(target[0].numpy(), k=cfg.k, tau=0.5, stride=cfg.output_stride)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
axes[0].imshow(img.clip(0, 1))
axes[0].set_title(f"augmented tile ({n_pts} points)")
axes[1].imshow(target[0], cmap="hot")
axes[1].set_title(f"target (stride {cfg.output_stride}, sigma {cfg.sigma})")
axes[2].imshow(img.clip(0, 1))
axes[2].scatter(pts[:, 0], pts[:, 1], s=45, facecolors="none", edgecolors="cyan")
axes[2].set_title(f"decoded back from target: {len(pts)}")
for ax in axes:
    ax.axis("off")

In [ ]:
# Target -> decode round trip over the val split: checks sigma / k / stride
# geometry is self-consistent before any training.
tot_gt = tot_dec = tot_tp = 0
for r in val_recs:
    gt = np.array([[p.x, p.y] for p in r.points], np.float32).reshape(-1, 2)
    out_h = (r.height + 31) // 32 * 32 // cfg.output_stride
    out_w = (r.width + 31) // 32 * 32 // cfg.output_stride
    t = render_targets(gt / cfg.output_stride, (out_h, out_w), cfg.sigma)
    dec, _ = decode_peaks(t, k=cfg.k, tau=0.5, stride=cfg.output_stride)
    tp, _, _ = match_points(dec, gt, radius_px=2.5 * cfg.output_stride)
    tot_gt += len(gt); tot_dec += len(dec); tot_tp += tp

print(f"GT {tot_gt} -> decoded {tot_dec}")
if tot_gt:
    print(f"recovered {100 * tot_tp / tot_gt:.2f}% | merge loss {100 * (tot_gt - tot_dec) / tot_gt:.2f}%")

## Train

AdamW on the decoder only (backbone frozen), warmup + cosine, bf16 autocast on CUDA. Per-epoch whole-image validation at `cfg.tau`; best checkpoint = lowest val loss.

In [ ]:
model, history, run_name, best_epoch = train(cfg)

#### Check

In [ ]:
# Load the best checkpoint from the run above.
run_dir = cfg.out_dir / run_name
print(f"loading {run_dir / 'best.pt'}")
model, _ = load_checkpoint(run_dir / "best.pt", device)
model.eval()

In [ ]:
# Alternatively, the last epoch's checkpoint instead of the best-val one:
# model, _ = load_checkpoint(run_dir / "last.pt", device)
# model.eval()

In [ ]:
tau_thresh = 0.35  # cfg.tau
k_kernel = cfg.k
rng_val = 7

val_ds = CropTileDataset(
    val_recs, cfg.val_images_dir, train=False, output_stride=cfg.output_stride,
    sigma=cfg.sigma, exclude_label_statuses=cfg.exclude_label_statuses,
)
n_show = min(3, len(val_ds))
picks = np.random.default_rng(rng_val).choice(len(val_ds), size=n_show, replace=False)

fig, axes = plt.subplots(n_show, 2, figsize=(15, 20 * n_show / 3), squeeze=False)
for row, idx in enumerate(picks):
    item = val_ds[int(idx)]
    with torch.no_grad(), torch.autocast(device.type, torch.bfloat16, enabled=device.type == "cuda"):
        logits = model(item["image"].unsqueeze(0).to(device))
    prob = torch.sigmoid(logits.float()).cpu()
    pred, _ = decode_peaks(
        prob, k=k_kernel, tau=tau_thresh, nms_radius=cfg.nms_radius, stride=cfg.output_stride
    )
    img = item["image"].permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    gt = item["points"]
    axes[row, 0].imshow(img.clip(0, 1))
    axes[row, 0].scatter(gt[:, 0], gt[:, 1], s=25, c="lime", alpha=0.8, label=f"GT {len(gt)}")
    axes[row, 0].scatter(pred[:, 0], pred[:, 1], s=60, facecolors="none",
                         edgecolors="red", label=f"pred {len(pred)}")
    axes[row, 0].legend(loc="upper right")
    axes[row, 0].set_title(item["name"][:70], fontsize=9)
    axes[row, 1].imshow(prob[0, 0], cmap="hot", vmin=0, vmax=1)
    axes[row, 1].set_title("predicted heatmap")
    for ax in axes[row]:
        ax.axis("off")

In [ ]:
def calibrate_tau(cfg, model, val_ds):
    """Calibrate the decode threshold: one forward pass over val, decoded
    at every tau. Picks tau minimising count MAE, sanity-checks F1 there."""
    val_loader = DataLoader(val_ds, batch_size=1, collate_fn=collate_val)
    taus = np.round(np.arange(0.05, 0.75, 0.05), 2)
    rows = sweep_tau(
        model, val_loader, device, taus, k=cfg.k, nms_radius=cfg.nms_radius,
        output_stride=cfg.output_stride, match_radius_px=cfg.match_radius_px,
    )

    best = min(rows, key=lambda r: r["count_mae"])
    fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
    ax1.plot([r["tau"] for r in rows], [r["count_mae"] for r in rows], "o-", color="#c0392b")
    ax1.set_xlabel("tau"); ax1.set_ylabel("count MAE", color="#c0392b")
    ax2 = ax1.twinx()
    ax2.plot([r["tau"] for r in rows], [r["f1"] for r in rows], "s-", color="#2b5f9e")
    ax2.set_ylabel("localization F1", color="#2b5f9e")
    ax1.axvline(best["tau"], color="gray", ls="--", lw=1)
    ax1.set_title(f"best tau {best['tau']:.2f}: MAE {best['count_mae']:.2f}, "
                f"F1 {best['f1']:.3f}, bias {best['count_bias']:+.2f}")
    return best, fig


best, fig = calibrate_tau(cfg, model, val_ds)
best